# Задание 4. Сборка генома de novo: Velvet, SPAdes и сравнение через QUAST

## Цель работы

Цель работы: выполнить сборку генома de novo с помощью Velvet при разных значениях k-mer, сравнить полученные сборки со сборкой SPAdes с помощью QUAST, а также попробовать улучшить параметры сборки и оценить изменения качества.

## Исходные данные

В работе использовались парные риды в формате FASTQ:

- `7_S4_L001_R1_001.fastq`
- `7_S4_L001_R2_001.fastq`


## Часть 1. Сборка с помощью Velvet

Были выбраны значения k-mer:

- 11
- 21
- 31



### SLURM-скрипт для запуска Velvet

```bash
#!/bin/bash
#SBATCH --job-name=velvet_kmers
#SBATCH --output=velvet_kmers_%j.out
#SBATCH --error=velvet_kmers_%j.err
#SBATCH --time=02:00:00
#SBATCH --cpus-per-task=1
#SBATCH --mem=4G

READ1=/home/STUDY/FBMF/bioinformatics/genome_de_novo/7_S4_L001_R1_001.fastq
READ2=/home/STUDY/FBMF/bioinformatics/genome_de_novo/7_S4_L001_R2_001.fastq
OUTDIR=~/seminars/sem11/genome_assembly_results/velvet

mkdir -p ${OUTDIR}

for K in 31 41 51 61
do
    mkdir -p ${OUTDIR}/k${K}

    /home/STUDY/FBMF/bioinformatics/soft/velvet/velveth \
    ${OUTDIR}/k${K} \
    ${K} \
    -fastq \
    -shortPaired \
    ${READ1} \
    ${READ2}

    /home/STUDY/FBMF/bioinformatics/soft/velvet/velvetg \
    ${OUTDIR}/k${K} \
    -ins_length 300
done
```


Для части 2:

```markdown
## Часть 2. Сравнение сборок с помощью QUAST

Для сравнения качества сборок использовалась программа QUAST. В анализ были включены сборки Velvet с разными значениями k-mer и сборка SPAdes.



### Скриншоты QUAST-отчёта для сравнения сборок



Ниже приведены скриншоты QUAST-отчёта, использованного для сравнения сборок Velvet и SPAdes.

**Скриншот 1. Основная таблица QUAST-отчёта.**

![QUAST part 2 screenshot 1](screenshots/part_2/part_2_1.png)



![QUAST part 2 screenshot 2](screenshots/part_2/part_2_2.png)



![QUAST part 2 screenshot 3](screenshots/part_2/part_2_3.png)



![QUAST part 2 screenshot 4](screenshots/part_2/part_2_4.png)


### Вывод по части 2

По результатам сравнения сборок через QUAST видно, что SPAdes показал более качественную и цельную сборку по сравнению с Velvet. У SPAdes получилось существенно меньшее число контигов: 49 против сотен и тысяч контигов у сборок Velvet. Это говорит о меньшей фрагментированности сборки.

Также у SPAdes наблюдается наибольшая длина самого крупного контига, у Velvet максимальный показатель оказался ниже. Значение N50 у SPAdes составило 440 п.н., что также выше, чем у сборок Velvet. Это означает, что значительная часть сборки SPAdes представлена более длинными фрагментами.

Для Velvet заметна сильная зависимость качества от выбранного k-mer. При малых значениях k-mer сборка получается сильно фрагментированной. При увеличении k-mer до 31 качество Velvet улучшается
SPAdes обычно даёт более устойчивую сборку, так как использует несколько k-mer и включает этапы коррекции ошибок. Velvet работает с выбранным k-mer, поэтому при неудачном значении параметра сборка может быть более фрагментированной.

## Часть 3. Улучшение сборки

Для улучшения сборки были изменены параметры запуска SPAdes и Velvet.

Для SPAdes был использован режим `--careful` и явно задан набор k-mer. Такой подход может уменьшить количество ошибок.

Для Velvet была выбрана сборка с промежуточным значением k-mer и добавлены параметры автоматической оценки покрытия: `-exp_cov auto` и `-cov_cutoff auto`. Эти параметры позволяют отфильтровать низкопокрытые участки, которые могут быть связаны с ошибками секвенирования.

### Скриншоты QUAST-отчёта для улучшенных сборок



Ниже приведены скриншоты QUAST-отчёта для сравнения исходных и улучшенных сборок SPAdes и Velvet.

**Скриншот 1. Основная таблица QUAST-отчёта для улучшенных сборок.**

![QUAST part 3 screenshot 1](screenshots/part_3/part_3_1.png)


![QUAST part 3 screenshot 2](screenshots/part_3/part_3_2.png)



![QUAST part 3 screenshot 3](screenshots/part_3/part_3_3.png)



![QUAST part 3 screenshot 4](screenshots/part_3/part_3_4.png)

### Вывод по части 3

После изменения параметров сборки результаты изменились по-разному для SPAdes и Velvet. Для SPAdes существенного улучшения не наблюдается: число контигов уменьшилось незначительно, с 49 до 47, а значения Largest contig и N50 остались практически теми же. Это говорит о том, что исходная сборка SPAdes уже была достаточно стабильной

Для Velvet улучшенная сборка на первый взгляд показывает заметное улучшение отдельных метрик. Количество контигов уменьшилось с 541 до 13, N50 увеличился со 104 до 243, а L50 снизился со 185 до 6. Это может указывать на снижение фрагментированности сборки.

Однако важно учитывать, что улучшение Velvet было связано с фильтрацией коротких контигов. Из-за этого из результата были удалены многие мелкие фрагменты, что резко уменьшило общую длину сборки.Поэтому улучшение некоторых метрик не означает однозначного повышения качества всей сборки. Скорее, сборка стала менее фрагментированной, но при этом потеряла значительную часть последовательностей.



## Общий вывод

В ходе работы были рассмотрены сборки генома de novo, выполненные с помощью Velvet и SPAdes, а их качество было оценено с использованием QUAST. Сравнение показало, что SPAdes даёт более устойчивую и цельную сборку: у него меньше контигов, выше N50 и больше длина самого крупного контига.

Velvet оказался более чувствителен к выбору параметров, особенно к размеру k-mer. При увеличении k-mer качество сборки Velvet улучшалось, однако даже лучший вариант уступал SPAdes по основным показателям непрерывности.

Попытка улучшения сборки показала, что для SPAdes изменение параметров почти не повлияло на результат, так как исходная сборка уже была достаточно стабильной. Для Velvet улучшение привело к уменьшению числа контигов и росту N50, но одновременно резко снизилась общая длина сборки. Это показывает, что улучшение отдельных метрик не всегда означает улучшение всей сборки: важно учитывать не только N50 и число контигов, но и полноту результата.

В целом наиболее надёжной сборкой в данной работе можно считать сборку SPAdes.